In [1]:
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm

# --- Step 1: Load Data ---
df = pd.read_csv("data/billboard_impact.csv")

# --- Step 2: Create Interaction Term ---
# poa = 1 for treatment group (Porto Alegre), 0 otherwise
# jul = 1 for post-intervention (July), 0 otherwise
df['did_interaction'] = df['poa'] * df['jul']

# --- Step 3: Specify the DiD Formula ---
# Includes fixed effects for group (poa), time (jul), and their interaction
formula = "deposits ~ did_interaction + C(poa) + C(jul)"

# --- Step 4: Fit the Model ---
model = smf.ols(formula=formula, data=df)
results = model.fit()

# --- Step 5: Extract and Print DiD Estimate ---
coef = results.params['did_interaction']
conf_int = results.conf_int().loc['did_interaction']
stderr = results.bse['did_interaction']
pval = results.pvalues['did_interaction']

print("=== Difference-in-Differences Estimation ===")
print(f"Treatment effect (DiD estimate): {coef:.2f}")
print(f"Standard error: {stderr:.2f}")
print(f"95% CI: ({conf_int[0]:.2f}, {conf_int[1]:.2f})")
print(f"P-value: {pval:.4f}")
print("\nModel Summary:")
print(results.summary())


=== Difference-in-Differences Estimation ===
Treatment effect (DiD estimate): 6.52
Standard error: 5.73
95% CI: (-4.71, 17.76)
P-value: 0.2548

Model Summary:
                            OLS Regression Results                            
Dep. Variable:               deposits   R-squared:                       0.313
Model:                            OLS   Adj. R-squared:                  0.312
Method:                 Least Squares   F-statistic:                     696.7
Date:                Tue, 02 Sep 2025   Prob (F-statistic):               0.00
Time:                        19:42:11   Log-Likelihood:                -26973.
No. Observations:                4600   AIC:                         5.395e+04
Df Residuals:                    4596   BIC:                         5.398e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t   

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
from auto_causal import run_causal_analysis

# Run causal analysis with a simple question
result = run_causal_analysis(
    query="Did the introduction of a billboard advertising campaign in Porto Alegre in July causally increase the number of deposits?",
    dataset_path="data/billboard_impact.csv",
    dataset_description="""The dataset billboard_impact.csv details information from a quasi-experiment assessing the influence of billboards on bank deposits in two cities: Porto Alegre (treatment group) and Florianopolis (control group). The csv file contains records with three variables: deposits (average bank deposits in Brazilian Reais), poa (A dummy indicator for the city of Porto Alegre. When it is zero, it means the samples are from Florianopolis.), and jul (A dummy for the month of July, or for the post intervention period. When it is zero it refers to samples from May, the pre-intervention period)
    """
)
print(result)
print(f"Causal effect: {result['results']['results']['effect_estimate']}")
print(f"Method used: {result['results']['results']['method_used']}")

2025-09-02 19:44:48,925 - INFO - Starting causal analysis run...
2025-09-02 19:44:48,928 - INFO - Initializing LLM client: Provider='together', Model='meta-llama/Llama-4-Scout-17B-16E-Instruct'
2025-09-02 19:44:48,979 - INFO - Constructed input for agent: 
My question is: Did the introduction of a billboard advertising campaign in Porto Alegre in July causally increase the number of deposits?
The dataset is located at: data/billboard_impact.csv
Dataset Description: The dataset billboard_impact.csv details information from a quasi-experiment assessing the influence of billboards on bank deposits in two cities: Porto Alegre (treatment group) and Florianopolis (control group). The csv file contains records with three variables: deposits (average bank deposits in Brazilian Reais), poa (A dummy indicator for the city of Porto Alegre. When it is zero, it means the samples are from Florianopolis.), and jul (A dummy for the month of July, or for the post intervention period. When it is zero it

[HumanMessage(content='\nAnalyze the following causal query **strictly in the context of the provided dataset information (if available)**. Identify the query type, key variables (mapping query terms to actual column names when possible), constraints, and any explicitly mentioned dataset path.\n\nUser Query: "Did the introduction of a billboard advertising campaign in Porto Alegre in July causally increase the number of deposits?"\n\nNo dataset context provided.\n\n# Add specific guidance for query types\nGuidance for Identifying Query Type:\n- EFFECT_ESTIMATION: Look for keywords like \'effect\', \'impact\', \'influence\', \'cause\', \'affect\', \'consequence\'. Also consider questions asking "how does X affect Y?" or comparing outcomes between groups based on an intervention.\n- COUNTERFACTUAL: Look for hypothetical scenarios, often using phrases like \'what if\', \'if X had been\', \'would Y have changed\', \'imagine if\', \'counterfactual\'.\n- CORRELATION: Look for keywords like \

2025-09-02 19:45:06,540 - INFO - HTTP Request: POST https://api.together.xyz/v1/chat/completions "HTTP/1.1 200 OK"
2025-09-02 19:45:06,542 - INFO - LLM identified time='jul', unit='poa'
2025-09-02 19:45:06,543 - INFO - Prioritizing LLM identified time variable: jul
2025-09-02 19:45:06,544 - INFO - Prioritizing LLM identified unit variable: poa
2025-09-02 19:45:06,548 - INFO - Panel data detected: Time='jul', Unit='poa', Periods=2, Units=2
2025-09-02 19:45:06,549 - INFO - Using LLM to identify potential treatment and outcome variables
2025-09-02 19:45:13,391 - INFO - HTTP Request: POST https://api.together.xyz/v1/chat/completions "HTTP/1.1 200 OK"
2025-09-02 19:45:13,392 - ERROR - Error in LLM identification: Extra data: line 5 column 1 (char 83)
Traceback (most recent call last):
  File "/home/sam/ethz/projects/causal-agent/auto_causal/components/dataset_analyzer.py", line 372, in _identify_potential_variables
    result = json.loads(json_match.group(0))
  File "/usr/lib64/python3.13/j

--------------------------
Validation result: {'valid': True, 'concerns': ['No evidence of parallel trends, which is a key assumption for difference-in-differences'], 'alternative_suggestions': ['synthetic_control'], 'recommended_method': 'difference_in_differences', 'assumptions': ['parallel trends between treatment and control groups before treatment', 'no spillover effects between groups', 'no anticipation effects before treatment', 'stable composition of treatment and control groups', 'treatment timing is exogenous']}
--------------------------


2025-09-02 19:46:06,961 - INFO - HTTP Request: POST https://api.together.xyz/v1/chat/completions "HTTP/1.1 200 OK"
2025-09-02 19:46:06,963 - ERROR - LLM fallback for time variable failed unexpectedly: 'NoneType' object has no attribute 'time_variable_name'
Traceback (most recent call last):
  File "/home/sam/ethz/projects/causal-agent/auto_causal/methods/difference_in_differences/llm_assist.py", line 66, in identify_time_variable
    llm_identified_col = parsed_result.time_variable_name
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'NoneType' object has no attribute 'time_variable_name'
2025-09-02 19:46:06,973 - WARNING - Could not identify time variable using heuristics or LLM fallback.
2025-09-02 19:46:06,975 - INFO - Attempting LLM call to identify group/unit variable...
2025-09-02 19:46:11,852 - INFO - HTTP Request: POST https://api.together.xyz/v1/chat/completions "HTTP/1.1 200 OK"
2025-09-02 19:46:11,857 - ERROR - LLM call for group/unit variable faile

summary_dict: {'query': 'Did the introduction of a billboard advertising campaign in Porto Alegre in July causally increase the number of deposits?', 'method_used': None, 'causal_effect': -38952690253346.39, 'standard_error': 26139592267161.965, 'confidence_interval': [-90185349667545.53, 12279969160852.758]}
CURRENT_OUTPUT_LOG_FILE: None
{'query': 'Did the introduction of a billboard advertising campaign in Porto Alegre in July causally increase the number of deposits?', 'method': 'difference_in_differences', 'results': {'results': {'effect_estimate': -38952690253346.39, 'confidence_interval': [-90185349667545.53, 12279969160852.758], 'standard_error': 26139592267161.965, 'p_value': 0.13617698757203303, 'method_used': 'difference_in_differences', 'llm_assumption_check': None, 'raw_results': None, 'diagnostics': {'parallel_trends': {'valid': True, 'p_value': 1.0, 'details': 'Insufficient pre-treatment data or variation to perform test. Defaulting to assuming parallel trends (unable to 

In [6]:
print(f"Estimate effect: {result['results']['results']['effect_estimate']}")
print(f"Method used: {result['results']['results']['method_used']}")

Estimate effect: -38952690253346.39
Method used: difference_in_differences
